# LLaMA2-7B Layer-16 `down_proj` G16/BFP4 PE Trace

This self-contained Colab notebook extracts the full real LLaMA2-7B trace and writes cycle-scheduled inputs for the baseline BFP PE. BFP4 uses an integer M3 magnitude and a shared biased-E5 exponent that directly encodes the mantissa-LSB scale exponent. RTL golden generation, simulation, and power analysis are not included.

Before running, select a GPU runtime and add `HF_TOKEN` to Colab Secrets. The Hugging Face account must have access to `meta-llama/Llama-2-7b-hf`. A 24 GB GPU such as L4, or an A100, is recommended; a 16 GB T4 may run out of memory.


## 1. Install dependencies and inspect the GPU

In [ ]:
%pip install -q "transformers==5.13.1" "datasets==4.0.0" accelerate sentencepiece numpy

import subprocess
subprocess.run(['nvidia-smi'], check=True)

## 2. Define the integrated G16/BFP4 trace tools

The next two cells contain the complete quantizer, packer, model wrapper, hook, and trace writer. No local Python modules are created or imported.

In [ ]:
#!/usr/bin/env python3
"""G16/BFP4 quantization and cycle-trace generation for the baseline BFP PE."""

from __future__ import annotations

import argparse
import csv
import hashlib
import json
import tempfile
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn.functional as F


@dataclass(frozen=True)
class TraceConfig:
    group_size: int = 16
    exponent_bits: int = 5
    exponent_bias: int = 15
    mantissa_bits: int = 3
    rounding: str = "nearest_even"

    @property
    def exponent_min(self) -> int:
        return -self.exponent_bias

    @property
    def exponent_max(self) -> int:
        return (1 << self.exponent_bits) - 1 - self.exponent_bias

    @property
    def mantissa_max(self) -> int:
        return (1 << self.mantissa_bits) - 1

    def validate(self) -> None:
        if self.group_size != 16:
            raise ValueError("This trace contract is frozen to GSIZE=16.")
        if self.exponent_bits != 5 or self.exponent_bias != 15:
            raise ValueError("This trace contract is frozen to biased E5 with bias 15.")
        if self.mantissa_bits != 3:
            raise ValueError("BFP4 requires one sign bit and three magnitude bits.")
        if self.rounding != "nearest_even":
            raise ValueError("The trace contract uses torch.round (nearest, ties-to-even).")


@dataclass(frozen=True)
class EncodedRows:
    encoded_exponent: torch.Tensor
    sign: torch.Tensor
    magnitude: torch.Tensor
    dequantized: torch.Tensor


def _reshape_blocks(rows: torch.Tensor, config: TraceConfig) -> tuple[torch.Tensor, int, int]:
    if not rows.is_floating_point():
        raise TypeError("rows must be a floating-point tensor.")
    width = rows.shape[-1]
    padding = (-width) % config.group_size
    flat = rows.reshape(-1, width).float()
    if padding:
        flat = F.pad(flat, (0, padding))
    return flat.reshape(flat.shape[0], -1, config.group_size), width, padding


def encode_rows(rows: torch.Tensor, config: TraceConfig) -> EncodedRows:
    """Encode rows using integer M3 and a biased-E5 mantissa-LSB scale exponent."""
    config.validate()
    blocks, width, padding = _reshape_blocks(rows, config)
    max_abs = blocks.abs().amax(dim=-1, keepdim=True)
    safe_max = max_abs.clamp_min(torch.finfo(torch.float32).tiny)
    scale_exponent = torch.floor(torch.log2(safe_max)) - (config.mantissa_bits - 1)
    scale_exponent = scale_exponent.clamp(config.exponent_min, config.exponent_max)
    scale_exponent = torch.where(max_abs == 0, torch.zeros_like(scale_exponent), scale_exponent)

    step = torch.pow(2.0, scale_exponent)
    signed_mantissa = torch.round(blocks / step)
    signed_mantissa = signed_mantissa.clamp(-config.mantissa_max, config.mantissa_max)
    signed_mantissa = signed_mantissa.to(torch.int16)

    sign = (signed_mantissa < 0).to(torch.uint8)
    magnitude = signed_mantissa.abs().to(torch.uint8)
    encoded_exponent = (scale_exponent.squeeze(-1).to(torch.int16) + config.exponent_bias).to(torch.uint8)

    dequantized = signed_mantissa.float() * step
    flat_dequantized = dequantized.reshape(dequantized.shape[0], -1)
    if padding:
        flat_dequantized = flat_dequantized[:, :-padding]
    dequantized_rows = flat_dequantized.reshape(rows.shape).to(rows.dtype)

    return EncodedRows(
        encoded_exponent=encoded_exponent,
        sign=sign,
        magnitude=magnitude,
        dequantized=dequantized_rows,
    )


def fake_quantize_rows(rows: torch.Tensor, config: TraceConfig) -> torch.Tensor:
    """Fake-quantize rows while retaining the original tensor shape and dtype."""
    config.validate()
    blocks, width, padding = _reshape_blocks(rows, config)
    max_abs = blocks.abs().amax(dim=-1, keepdim=True)
    safe_max = max_abs.clamp_min(torch.finfo(torch.float32).tiny)
    scale_exponent = torch.floor(torch.log2(safe_max)) - (config.mantissa_bits - 1)
    scale_exponent = scale_exponent.clamp(config.exponent_min, config.exponent_max)
    scale_exponent = torch.where(max_abs == 0, torch.zeros_like(scale_exponent), scale_exponent)
    step = torch.pow(2.0, scale_exponent)
    signed_mantissa = torch.round(blocks / step)
    signed_mantissa = signed_mantissa.clamp(-config.mantissa_max, config.mantissa_max)
    flat_dequantized = (signed_mantissa * step).reshape(blocks.shape[0], -1)
    if padding:
        flat_dequantized = flat_dequantized[:, :-padding]
    return flat_dequantized.reshape(rows.shape).to(rows.dtype)


def pack_sign(sign: torch.Tensor) -> np.ndarray:
    lanes = sign.shape[-1]
    powers = torch.bitwise_left_shift(
        torch.ones(lanes, dtype=torch.int64, device=sign.device),
        torch.arange(lanes, dtype=torch.int64, device=sign.device),
    )
    return (sign.to(torch.int64) * powers).sum(dim=-1).cpu().numpy().astype(np.uint16)


def pack_magnitude(magnitude: torch.Tensor, mantissa_bits: int) -> np.ndarray:
    lanes = magnitude.shape[-1]
    shifts = torch.arange(lanes, dtype=torch.int64, device=magnitude.device) * mantissa_bits
    powers = torch.bitwise_left_shift(
        torch.ones(lanes, dtype=torch.int64, device=magnitude.device), shifts
    )
    return (magnitude.to(torch.int64) * powers).sum(dim=-1).cpu().numpy().astype(np.uint64)


def unpack_sign(packed: np.ndarray, lanes: int) -> np.ndarray:
    shifts = np.arange(lanes, dtype=np.uint64)
    return ((packed.astype(np.uint64)[..., None] >> shifts) & 1).astype(np.uint8)


def unpack_magnitude(packed: np.ndarray, lanes: int, mantissa_bits: int) -> np.ndarray:
    shifts = np.arange(lanes, dtype=np.uint64) * mantissa_bits
    mask = (1 << mantissa_bits) - 1
    return ((packed.astype(np.uint64)[..., None] >> shifts) & mask).astype(np.uint8)


def _hex_line(
    acc_clear: int,
    weight_load: int,
    in_valid: int,
    w_sign: int,
    w_exp: int,
    w_magnitude: int,
    a_sign: int,
    a_exp: int,
    a_magnitude: int,
) -> str:
    return (
        f"{acc_clear:x} {weight_load:x} {in_valid:x} "
        f"{w_sign:04x} {w_exp:02x} {w_magnitude:012x} "
        f"{a_sign:04x} {a_exp:02x} {a_magnitude:012x}\n"
    )


def _validate_encoded(name: str, encoded: EncodedRows, config: TraceConfig) -> None:
    if encoded.sign.shape[-1] != config.group_size:
        raise AssertionError(f"{name}: incorrect group size.")
    if int(encoded.encoded_exponent.min()) < 0 or int(encoded.encoded_exponent.max()) >= (1 << config.exponent_bits):
        raise AssertionError(f"{name}: encoded exponent is outside E5.")
    if int(encoded.magnitude.max()) > config.mantissa_max:
        raise AssertionError(f"{name}: magnitude is outside M3.")

    sign_packed = pack_sign(encoded.sign)
    magnitude_packed = pack_magnitude(encoded.magnitude, config.mantissa_bits)
    sign_roundtrip = unpack_sign(sign_packed, config.group_size)
    magnitude_roundtrip = unpack_magnitude(
        magnitude_packed, config.group_size, config.mantissa_bits
    )
    np.testing.assert_array_equal(sign_roundtrip, encoded.sign.cpu().numpy())
    np.testing.assert_array_equal(magnitude_roundtrip, encoded.magnitude.cpu().numpy())


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def write_trace_artifacts(
    output_dir: Path,
    raw_weights: torch.Tensor,
    raw_activations: torch.Tensor,
    channel_indices: list[int],
    token_positions: list[int],
    token_ids: list[int],
    metadata: dict[str, Any],
    config: TraceConfig,
) -> dict[str, Any]:
    """Write raw tensors, packed logical blocks, and PE-scheduled input.dat."""
    config.validate()
    output_dir.mkdir(parents=True, exist_ok=True)

    if raw_weights.ndim != 2 or raw_activations.ndim != 2:
        raise ValueError("raw_weights and raw_activations must both be rank-2 tensors.")
    if raw_weights.shape[0] != len(channel_indices):
        raise ValueError("channel_indices does not match raw_weights rows.")
    if raw_activations.shape[0] != len(token_positions):
        raise ValueError("token_positions does not match raw_activations rows.")
    if len(token_ids) != len(token_positions):
        raise ValueError("token_ids does not match token_positions.")
    if raw_weights.shape[1] != raw_activations.shape[1]:
        raise ValueError("weight and activation K dimensions differ.")
    if raw_weights.shape[1] % config.group_size:
        raise ValueError("The K dimension must be divisible by GSIZE=16.")

    weights = raw_weights.detach().cpu().contiguous()
    activations = raw_activations.detach().cpu().contiguous()
    encoded_w = encode_rows(weights, config)
    encoded_a = encode_rows(activations, config)
    _validate_encoded("weight", encoded_w, config)
    _validate_encoded("activation", encoded_a, config)

    w_sign = pack_sign(encoded_w.sign)
    w_magnitude = pack_magnitude(encoded_w.magnitude, config.mantissa_bits)
    a_sign = pack_sign(encoded_a.sign)
    a_magnitude = pack_magnitude(encoded_a.magnitude, config.mantissa_bits)
    w_exp = encoded_w.encoded_exponent.cpu().numpy().astype(np.uint8)
    a_exp = encoded_a.encoded_exponent.cpu().numpy().astype(np.uint8)

    token_count = activations.shape[0]
    channel_count = weights.shape[0]
    block_count = weights.shape[1] // config.group_size
    dot_product_count = token_count * channel_count
    total_cycles = dot_product_count * (block_count + 1)

    raw_path = output_dir / "raw_tensor.pt"
    torch.save(
        {
            "weight": weights,
            "activation": activations,
            "channel_indices": channel_indices,
            "token_positions": token_positions,
            "token_ids": token_ids,
        },
        raw_path,
    )

    quantized_path = output_dir / "quantized_tensor.npz"
    np.savez_compressed(
        quantized_path,
        weight_exponent=w_exp,
        weight_sign=encoded_w.sign.cpu().numpy(),
        weight_magnitude=encoded_w.magnitude.cpu().numpy(),
        weight_dequantized=encoded_w.dequantized.float().cpu().numpy(),
        activation_exponent=a_exp,
        activation_sign=encoded_a.sign.cpu().numpy(),
        activation_magnitude=encoded_a.magnitude.cpu().numpy(),
        activation_dequantized=encoded_a.dequantized.float().cpu().numpy(),
    )

    token_slot = np.repeat(np.arange(token_count, dtype=np.int16), channel_count * block_count)
    channel_slot = np.tile(
        np.repeat(np.arange(channel_count, dtype=np.int16), block_count), token_count
    )
    block_index = np.tile(np.arange(block_count, dtype=np.int16), dot_product_count)
    logical_path = output_dir / "logical_blocks.npz"
    np.savez_compressed(
        logical_path,
        token_slot=token_slot,
        channel_slot=channel_slot,
        block_index=block_index,
        weight_sign=np.broadcast_to(w_sign[None, :, :], (token_count, channel_count, block_count)).reshape(-1),
        weight_exponent=np.broadcast_to(w_exp[None, :, :], (token_count, channel_count, block_count)).reshape(-1),
        weight_magnitude=np.broadcast_to(w_magnitude[None, :, :], (token_count, channel_count, block_count)).reshape(-1),
        activation_sign=np.broadcast_to(a_sign[:, None, :], (token_count, channel_count, block_count)).reshape(-1),
        activation_exponent=np.broadcast_to(a_exp[:, None, :], (token_count, channel_count, block_count)).reshape(-1),
        activation_magnitude=np.broadcast_to(a_magnitude[:, None, :], (token_count, channel_count, block_count)).reshape(-1),
    )

    input_path = output_dir / "input.dat"
    index_path = output_dir / "trace_index.csv"
    cycle = 0
    with input_path.open("w", encoding="ascii", newline="\n") as input_file, index_path.open(
        "w", encoding="utf-8", newline=""
    ) as index_file:
        index_writer = csv.writer(index_file)
        index_writer.writerow(
            ["token_slot", "token_position", "token_id", "channel_slot", "output_channel", "setup_cycle", "first_valid_cycle", "final_cycle"]
        )
        for token_slot_index in range(token_count):
            for channel_slot_index in range(channel_count):
                setup_cycle = cycle
                input_file.write(
                    _hex_line(
                        1, 1, 0,
                        int(w_sign[channel_slot_index, 0]),
                        int(w_exp[channel_slot_index, 0]),
                        int(w_magnitude[channel_slot_index, 0]),
                        0, config.exponent_bias, 0,
                    )
                )
                cycle += 1
                for block in range(block_count):
                    next_weight_block = min(block + 1, block_count - 1)
                    weight_load = int(block + 1 < block_count)
                    input_file.write(
                        _hex_line(
                            0, weight_load, 1,
                            int(w_sign[channel_slot_index, next_weight_block]),
                            int(w_exp[channel_slot_index, next_weight_block]),
                            int(w_magnitude[channel_slot_index, next_weight_block]),
                            int(a_sign[token_slot_index, block]),
                            int(a_exp[token_slot_index, block]),
                            int(a_magnitude[token_slot_index, block]),
                        )
                    )
                    cycle += 1
                index_writer.writerow(
                    [
                        token_slot_index,
                        token_positions[token_slot_index],
                        token_ids[token_slot_index],
                        channel_slot_index,
                        channel_indices[channel_slot_index],
                        setup_cycle,
                        setup_cycle + 1,
                        cycle - 1,
                    ]
                )

    if cycle != total_cycles:
        raise AssertionError(f"Scheduled {cycle} cycles, expected {total_cycles}.")

    complete_metadata = {
        **metadata,
        "trace_format": "baseline_bfp_pe_cycle_input_v2",
        "bfp_config": asdict(config),
        "bfp_format": "BFP4 (1 sign + 3-bit integer magnitude, shared biased-E5 scale exponent)",
        "shared_exponent_semantics": "value = signed_integer_magnitude * 2**(encoded_exponent - exponent_bias)",
        "unbiased_exponent_range": [config.exponent_min, config.exponent_max],
        "zero_block_encoded_exponent": config.exponent_bias,
        "lane_packing": "lane 0 occupies the least-significant bits",
        "weight_timing": "in_valid uses the registered weight; same-cycle weight_load becomes active next cycle",
        "tensor_shapes": {
            "weight": list(weights.shape),
            "activation": list(activations.shape),
        },
        "token_positions": token_positions,
        "token_ids": token_ids,
        "output_channels": channel_indices,
        "blocks_per_dot_product": block_count,
        "cycles_per_dot_product": block_count + 1,
        "dot_product_count": dot_product_count,
        "valid_cycles": dot_product_count * block_count,
        "total_cycles": total_cycles,
    }
    metadata_path = output_dir / "trace_metadata.json"
    metadata_path.write_text(
        json.dumps(complete_metadata, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
    )

    weight_zero_blocks = int((encoded_w.magnitude.sum(dim=-1) == 0).sum())
    activation_zero_blocks = int((encoded_a.magnitude.sum(dim=-1) == 0).sum())
    summary = {
        "weight_zero_blocks": weight_zero_blocks,
        "activation_zero_blocks": activation_zero_blocks,
        "weight_encoded_exponent_histogram": {
            str(key): int(value)
            for key, value in zip(*np.unique(w_exp, return_counts=True))
        },
        "activation_encoded_exponent_histogram": {
            str(key): int(value)
            for key, value in zip(*np.unique(a_exp, return_counts=True))
        },
        "input_dat_lines": total_cycles,
        "logical_block_rows": dot_product_count * block_count,
        "pack_unpack_self_check": "pass",
    }
    summary_path = output_dir / "trace_summary.json"
    summary_path.write_text(
        json.dumps(summary, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
    )

    artifact_paths = [
        raw_path,
        quantized_path,
        logical_path,
        input_path,
        index_path,
        metadata_path,
        summary_path,
    ]
    checksum_path = output_dir / "checksums.sha256"
    checksum_path.write_text(
        "".join(f"{_sha256(path)}  {path.name}\n" for path in artifact_paths),
        encoding="ascii",
    )
    return complete_metadata


def run_self_test() -> None:
    config = TraceConfig()
    weights = torch.tensor(
        [[0.0, -1.0, 0.5, 1.5, 0.25, -0.25, 2.0, -2.0] * 2], dtype=torch.float32
    )
    activations = torch.tensor(
        [[1.0, 0.0, -0.5, 0.5, -1.5, 0.25, 0.0, 2.0] * 2], dtype=torch.float32
    )
    with tempfile.TemporaryDirectory() as temp_dir:
        metadata = write_trace_artifacts(
            Path(temp_dir), weights, activations, [0], [128], [42],
            {"self_test": True}, config,
        )
        lines = (Path(temp_dir) / "input.dat").read_text(encoding="ascii").splitlines()
        if len(lines) != 2:
            raise AssertionError("G16 self-test must produce one setup and one valid cycle.")
        if metadata["valid_cycles"] != 1 or metadata["total_cycles"] != 2:
            raise AssertionError("Self-test cycle accounting failed.")
        setup_fields = lines[0].split()
        valid_fields = lines[1].split()
        if setup_fields[4] != "0e" or valid_fields[7] != "0e":
            raise AssertionError("B-convention self-test expected scale exponent -1 encoded as E5=14.")
    print("[PASS] G16/BFP4 B-convention quantization, pack/unpack, and scheduling self-test")


In [ ]:
#!/usr/bin/env python3
"""Extract a real LLaMA2-7B Layer-16 down_proj G16/BFP4 PE trace."""

from __future__ import annotations

import argparse
import gc
import hashlib
import json
import os
import platform
import shutil
from getpass import getpass
from pathlib import Path

import datasets
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer



MODEL_ID = "meta-llama/Llama-2-7b-hf"
DATASET_ID = "Salesforce/wikitext"
DATASET_CONFIG = "wikitext-2-raw-v1"
DATASET_SPLIT = "test"
TARGET_LAYER = 16
TARGET_MODULE = "model.layers.16.mlp.down_proj"


class CaptureComplete(RuntimeError):
    pass


class BFPLinear(nn.Module):
    def __init__(self, linear: nn.Linear, config: TraceConfig):
        super().__init__()
        self.linear = linear
        self.config = config

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        quantized_inputs = fake_quantize_rows(inputs, self.config)
        return F.linear(
            quantized_inputs, self.linear.weight, self.linear.bias
        ).to(torch.float16)


@torch.no_grad()
def quantize_weight_in_place(
    weight: torch.Tensor, config: TraceConfig, chunk_rows: int = 128
) -> None:
    for start in range(0, weight.shape[0], chunk_rows):
        end = min(start + chunk_rows, weight.shape[0])
        weight[start:end].copy_(fake_quantize_rows(weight[start:end], config))


@torch.no_grad()
def replace_linear_layers(module: nn.Module, config: TraceConfig) -> int:
    replaced = 0
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear):
            quantize_weight_in_place(child.weight, config)
            setattr(module, name, BFPLinear(child, config))
            replaced += 1
        else:
            replaced += replace_linear_layers(child, config)
    return replaced


def get_hf_token() -> str:
    token = os.getenv("HF_TOKEN")
    if token:
        return token
    try:
        from google.colab import userdata

        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
    if not token:
        token = getpass("HF_TOKEN: ")
    if not token:
        raise RuntimeError("HF_TOKEN is required for the gated LLaMA2 repository.")
    return token


def evenly_spaced_indices(size: int, count: int) -> list[int]:
    if count <= 0 or count > size:
        raise ValueError("channel_count must be between 1 and the output width.")
    if count == 1:
        return [0]
    indices = torch.linspace(0, size - 1, count).round().to(torch.int64).tolist()
    if len(set(indices)) != count:
        raise AssertionError("Evenly spaced channel selection produced duplicate indices.")
    return indices


def tensor_sha256(tensor: torch.Tensor) -> str:
    contiguous = tensor.detach().cpu().contiguous()
    return hashlib.sha256(contiguous.numpy().tobytes()).hexdigest()
@torch.inference_mode()
def extract_trace(args: argparse.Namespace) -> None:
    config = TraceConfig()
    config.validate()
    if not torch.cuda.is_available():
        raise RuntimeError("An NVIDIA CUDA GPU is required to extract this trace.")
    if args.token_start < 0 or args.token_count <= 0:
        raise ValueError("token_start must be nonnegative and token_count must be positive.")
    if args.token_start + args.token_count > args.context_length:
        raise ValueError("Selected token positions exceed the context length.")

    torch.manual_seed(0)
    torch.backends.cuda.matmul.allow_tf32 = False
    token = get_hf_token()

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=token)
    dataset = load_dataset(DATASET_ID, DATASET_CONFIG, split=DATASET_SPLIT)
    text = "\n\n".join(dataset["text"])
    all_input_ids = tokenizer(text, return_tensors="pt").input_ids
    if all_input_ids.shape[1] < args.context_length:
        raise RuntimeError("WikiText-2 does not contain one complete requested context.")
    context_ids = all_input_ids[:, : args.context_length].contiguous()
    token_positions = list(range(args.token_start, args.token_start + args.token_count))
    selected_token_ids = context_ids[0, token_positions].tolist()

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float16,
        device_map=0,
        attn_implementation="eager",
        token=token,
    )
    model.eval()
    model.config.use_cache = False

    target_linear = model.model.layers[TARGET_LAYER].mlp.down_proj
    if not isinstance(target_linear, nn.Linear):
        raise TypeError("Expected the target down_proj module to be nn.Linear before wrapping.")
    if target_linear.in_features % config.group_size:
        raise ValueError("Target K dimension is not divisible by GSIZE=16.")
    channel_indices = evenly_spaced_indices(target_linear.out_features, args.channel_count)
    raw_weights = target_linear.weight[channel_indices].detach().cpu().clone()

    replaced_count = 0
    for layer_index in range(TARGET_LAYER + 1):
        replaced_count += replace_linear_layers(model.model.layers[layer_index], config)
    wrapped_target = model.model.layers[TARGET_LAYER].mlp.down_proj
    if not isinstance(wrapped_target, BFPLinear):
        raise TypeError("Target down_proj was not wrapped as BFPLinear.")

    captured: dict[str, torch.Tensor] = {}

    def capture_target_input(_module: nn.Module, inputs: tuple[torch.Tensor, ...]) -> None:
        activation = inputs[0]
        if activation.ndim != 3 or activation.shape[0] != 1:
            raise RuntimeError(f"Unexpected target activation shape: {tuple(activation.shape)}")
        captured["activation"] = activation[0, token_positions].detach().cpu().clone()
        raise CaptureComplete

    hook = wrapped_target.register_forward_pre_hook(capture_target_input)
    try:
        model(context_ids.to(next(model.parameters()).device), use_cache=False)
    except CaptureComplete:
        pass
    finally:
        hook.remove()

    if "activation" not in captured:
        raise RuntimeError("The target down_proj input hook did not capture an activation.")
    raw_activations = captured["activation"]
    if raw_activations.shape[1] != raw_weights.shape[1]:
        raise RuntimeError("Captured activation and target weight K dimensions differ.")

    metadata = {
        "model": MODEL_ID,
        "model_revision": getattr(model.config, "_commit_hash", None),
        "dataset": f"{DATASET_ID}/{DATASET_CONFIG}",
        "dataset_split": DATASET_SPLIT,
        "dataset_fingerprint": getattr(dataset, "_fingerprint", None),
        "context_selection": "first complete tokenized context",
        "context_length": args.context_length,
        "target_module": TARGET_MODULE,
        "target_layer": TARGET_LAYER,
        "upstream_execution": "layers 0 through 16 use W/A G16 BFP4 fake quantization",
        "selected_weight_source": "original FP16 target weights, encoded by the trace quantizer",
        "selected_activation_source": "input to the wrapped target down_proj before local activation quantization",
        "linear_input_features": int(raw_weights.shape[1]),
        "linear_output_features": int(target_linear.out_features),
        "quantized_linear_modules_before_capture": replaced_count,
        "selection_policy": {
            "tokens": "contiguous positions",
            "channels": "evenly spaced across down_proj output channels",
        },
        "source_hashes": {
            "context_input_ids_sha256": tensor_sha256(context_ids),
            "selected_weight_fp16_sha256": tensor_sha256(raw_weights),
            "selected_activation_fp16_sha256": tensor_sha256(raw_activations),
        },
        "software": {
            "python": platform.python_version(),
            "pytorch": torch.__version__,
            "transformers": transformers.__version__,
            "datasets": datasets.__version__,
            "cuda": torch.version.cuda,
            "gpu": torch.cuda.get_device_name(0),
            "tokenizer_class": tokenizer.__class__.__name__,
        },
    }
    complete_metadata = write_trace_artifacts(
        args.output_dir,
        raw_weights,
        raw_activations,
        channel_indices,
        token_positions,
        selected_token_ids,
        metadata,
        config,
    )

    archive_path = None
    if not args.no_archive:
        archive_base = args.output_dir.parent / args.output_dir.name
        archive_path = Path(
            shutil.make_archive(str(archive_base), "zip", args.output_dir.parent, args.output_dir.name)
        )

    print(json.dumps(complete_metadata, indent=2, ensure_ascii=False))
    print(f"[PASS] Trace artifacts: {args.output_dir.resolve()}")
    if archive_path is not None:
        print(f"[PASS] Archive: {archive_path.resolve()}")

    del model, raw_weights, raw_activations, captured
    gc.collect()
    torch.cuda.empty_cache()



## 3. Run the G16/BFP4 synthetic self-test

In [ ]:
run_self_test()


## 4. Load the Hugging Face token securely

The token is copied into process memory and is never printed or saved in the trace.

In [ ]:
import os
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
except Exception as exc:
    raise RuntimeError('Add HF_TOKEN to Colab Secrets and enable notebook access.') from exc
if not hf_token:
    raise RuntimeError('Add HF_TOKEN to Colab Secrets and enable notebook access.')
os.environ['HF_TOKEN'] = hf_token
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
print('HF_TOKEN is available in kernel memory.')


## 5. Configure the full trace

This notebook is configured to extract the complete 16-token by 16-channel trace directly.

In [ ]:
CONTEXT_LENGTH = 2048
TOKEN_START = 128
TOKEN_COUNT = 16
CHANNEL_COUNT = 16
OUTPUT_DIR = 'outputs/llama2_layer16_down_proj_g16_bfp4'

print({
    'mode': 'full',
    'context_length': CONTEXT_LENGTH,
    'token_count': TOKEN_COUNT,
    'channel_count': CHANNEL_COUNT,
    'output_dir': OUTPUT_DIR,
})


## 6. Extract and package the trace

The extraction performs one model forward pass and schedules 176,384 PE cycles.

In [ ]:
args = argparse.Namespace(
    context_length=CONTEXT_LENGTH,
    token_start=TOKEN_START,
    token_count=TOKEN_COUNT,
    channel_count=CHANNEL_COUNT,
    output_dir=Path(OUTPUT_DIR),
    no_archive=False,
)
extract_trace(args)


## 7. Inspect metadata and download the ZIP archive

In [ ]:
import json
from pathlib import Path
from google.colab import files

output_path = Path(OUTPUT_DIR)
metadata = json.loads((output_path / 'trace_metadata.json').read_text(encoding='utf-8'))
summary = json.loads((output_path / 'trace_summary.json').read_text(encoding='utf-8'))
print(json.dumps({
    'tensor_shapes': metadata['tensor_shapes'],
    'blocks_per_dot_product': metadata['blocks_per_dot_product'],
    'dot_product_count': metadata['dot_product_count'],
    'valid_cycles': metadata['valid_cycles'],
    'total_cycles': metadata['total_cycles'],
    'pack_unpack_self_check': summary['pack_unpack_self_check'],
}, indent=2))

archive_path = output_path.with_suffix('.zip')
if not archive_path.exists():
    raise FileNotFoundError(archive_path)
files.download(str(archive_path))
